In [4]:
!pip install requests beautifulsoup4 pandas lxml deepface tf-keras fastapi uvicorn python-multipart pyngrok opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.7 MB/s eta 0:00:00


In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import random

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) Gecko/20100101 Firefox/124.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
]

def get_headers():
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-IN,en-GB;q=0.9,en-US;q=0.8",
        "Accept-Encoding": "gzip, deflate, br",
        "DNT": "1",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Referer": "https://www.amazon.in/",
    }

def create_session():
    session = requests.Session()
    print("🔗 Creating session with Amazon homepage...")
    try:
        session.get("https://www.amazon.in", headers=get_headers(), timeout=15)
        print("✅ Session created with cookies")
        time.sleep(random.uniform(2, 4))
    except Exception as e:
        print(f"Warning: {e}")
    return session

def fetch_page(session, url, max_retries=3):
    for attempt in range(1, max_retries + 1):
        try:
            response = session.get(url, headers=get_headers(), timeout=20)
            if response.status_code == 200:
                if "Enter the characters you see below" in response.text:
                    print(f"   ⚠️ CAPTCHA on attempt {attempt}")
                    time.sleep(random.uniform(5, 10))
                    continue
                return response
            elif response.status_code == 503:
                wait = random.uniform(5, 10) * attempt
                print(f"   503 error - attempt {attempt}/{max_retries}, waiting {wait:.0f}s...")
                time.sleep(wait)
            else:
                print(f"   HTTP {response.status_code}")
                time.sleep(3)
        except Exception as e:
            print(f"   Error: {e}")
            time.sleep(5)
    return None

def parse_products(html_content, page_number):
    soup = BeautifulSoup(html_content, "lxml")
    cards = soup.find_all("div", {"data-component-type": "s-search-result"})
    print(f"   ✅ Found {len(cards)} products on page {page_number}")
    products = []
    for card in cards:
        title_tag   = card.find("h2")
        price_tag   = card.find("span", class_="a-price-whole")
        rating_tag  = card.find("span", class_="a-icon-alt")
        img_tag     = card.find("img", class_="s-image")
        link_tag    = card.find("a", class_="a-link-normal s-no-outline")
        if not link_tag and card.find("h2"):
            link_tag = card.find("h2").find("a")
        sponsored = card.find(
            lambda t: t.name in ["span","div"] and t.string and "Sponsored" in t.string
        )
        products.append({
            "Title":      title_tag.get_text(strip=True) if title_tag else "N/A",
            "Price":      "₹" + price_tag.get_text(strip=True).replace(",","") if price_tag else "N/A",
            "Rating":     rating_tag.get_text(strip=True) if rating_tag else "N/A",
            "Image URL":  img_tag["src"] if img_tag else "N/A",
            "Product URL":"https://www.amazon.in" + link_tag["href"] if link_tag else "N/A",
            "Ad/Organic": "Ad (Sponsored)" if sponsored else "Organic",
            "Page":       page_number,
        })
    return products

def scrape_amazon_laptops(total_pages=3):
    print("="*55)
    print("  Amazon India - Laptop Scraper")
    print("="*55)
    session = create_session()
    all_products = []
    for page in range(1, total_pages + 1):
        url = f"https://www.amazon.in/s?k=laptops&page={page}"
        print(f"\n📄 Scraping page {page}/{total_pages}...")
        response = fetch_page(session, url)
        if response is None:
            print(f"   ❌ Failed page {page}, skipping.")
            continue
        products = parse_products(response.content, page)
        all_products.extend(products)
        if page < total_pages:
            delay = random.uniform(4, 8)
            print(f"   ⏳ Waiting {delay:.1f}s...")
            time.sleep(delay)
    return pd.DataFrame(all_products)

# ▶️ RUN SCRAPER
df = scrape_amazon_laptops(total_pages=3)
print(f"\n🎉 Total products: {len(df)}")

  Amazon India - Laptop Scraper
🔗 Creating session with Amazon homepage...
✅ Session created with cookies

📄 Scraping page 1/3...
   ✅ Found 16 products on page 1
   ⏳ Waiting 6.5s...

📄 Scraping page 2/3...
   ✅ Found 16 products on page 2
   ⏳ Waiting 4.0s...

📄 Scraping page 3/3...
   ✅ Found 16 products on page 3

🎉 Total products: 48


In [7]:
# Show the data as a table
df.head(20)

,Title,Price,Rating,Image URL,Product URL,Ad/Organic,Page
0,"Acer Smartchoice Aspire One, AMD Ryzen 3-7320U...",₹39990,5.0 out of 5 stars,https://m.media-amazon.com/images/I/71ouu-iX3p...,https://www.amazon.in/Acer-Smartchoice-3-7320U...,Organic,1
1,"Dell 15 (Previously Inspiron) Laptop, 14th Gen...",₹46490,4.1 out of 5 stars,https://m.media-amazon.com/images/I/717WZ7Wriw...,https://www.amazon.in/Dell-Previously-Inspiron...,Organic,1
2,"HP 15, 13th Gen Intel Core i3-1315U Laptop (8G...",₹50040,4.2 out of 5 stars,https://m.media-amazon.com/images/I/61R5Ecv7i-...,https://www.amazon.in/HP-i3-1315U-Anti-Glare-M...,Organic,1
3,"ASUS Vivobook 16,Smartchoice, 13th Gen, Intel ...",₹62990,4.2 out of 5 stars,https://m.media-amazon.com/images/I/71aaW7HXKn...,https://www.amazon.in/ASUS-Vivobook-i5-13420H-...,Organic,1
4,"ASUS Vivobook 15, Smartchoice, AMD Ryzen 7 582...",₹54990,4.0 out of 5 stars,https://m.media-amazon.com/images/I/71zMooVIVA...,https://www.amazon.in/ASUS-Vivobook-Smartchoic...,Organic,1
5,Lenovo V15 G4 AMD Athlon Silver 7120U Laptop 8...,₹42999,4.0 out of 5 stars,https://m.media-amazon.com/images/I/61AccNkmFF...,https://www.amazon.in/Lenovo-V15-Lifetime-Vali...,Organic,1
6,"MSI Modern 14, Intel 13th Gen. Core i3 1315U,1...",₹44990,N/A,https://m.media-amazon.com/images/I/61O737i7Qe...,https://www.amazon.in/MSI-Business-Windows-Cla...,Organic,1
7,𝗗𝗲𝗹𝗹Laptop Model 5420 | 𝗜𝗡𝗧𝗘𝗟i5 11th Gen Proce...,₹31990,5.0 out of 5 stars,https://m.media-amazon.com/images/I/51LHNjAjH9...,https://www.amazon.in/%F0%9D%97%97%F0%9D%97%B2...,Organic,1
8,Apple 2026 MacBook Neo 13″ Laptop with A18 Pro...,₹64990,4.7 out of 5 stars,https://m.media-amazon.com/images/I/61amETli1D...,https://www.amazon.in/Apple-2026-MacBook-Lapto...,Organic,1
9,"ASUS TUF A15 (2025), AMD Ryzen 7 7445HS,RTX 30...",₹68990,4.3 out of 5 stars,https://m.media-amazon.com/images/I/71K4CpdfmH...,https://www.amazon.in/ASUS-Upgradeable-Keyboar...,Organic,1


In [8]:
from datetime import datetime
from google.colab import files

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"amazon_laptops_{timestamp}.csv"
df.to_csv(filename, index=False, encoding="utf-8-sig")
print(f"✅ Saved: {filename}")
print(f"📦 Total rows: {len(df)}")

# Auto download to your PC
files.download(filename)

✅ Saved: amazon_laptops_20260604_081111.csv
📦 Total rows: 48


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
print("📊 Summary:")
print(f"  Total products : {len(df)}")
print(f"  Ads found      : {len(df[df['Ad/Organic']=='Ad (Sponsored)'])}")
print(f"  Organic results: {len(df[df['Ad/Organic']=='Organic'])}")
print(f"  With price     : {len(df[df['Price']!='N/A'])}")
print(f"  With rating    : {len(df[df['Rating']!='N/A'])}")
print(f"\n  Columns: {list(df.columns)}")

📊 Summary:
  Total products : 48
  Ads found      : 0
  Organic results: 48
  With price     : 48
  With rating    : 46

  Columns: ['Title', 'Price', 'Rating', 'Image URL', 'Product URL', 'Ad/Organic', 'Page']


In [10]:
!pip install deepface tf-keras fastapi uvicorn \
    python-multipart pyngrok opencv-python-headless numpy

In [11]:
%run task2_train.py

📦 Loading DeepFace library...
26-06-04 08:12:14 - Directory /root/.deepface has been created
26-06-04 08:12:14 - Directory /root/.deepface/weights has been created
✅ DeepFace imported successfully.

⬇️  Building FaceNet model (downloading weights if first time)...
    This may take 1-2 minutes on first run...

26-06-04 08:12:18 - 🔗 facenet_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facenet_weights.h5 to /root/.deepface/weights/facenet_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facenet_weights.h5
To: /root/.deepface/weights/facenet_weights.h5
100%|██████████| 92.2M/92.2M [00:00<00:00, 287MB/s]


✅ FaceNet model loaded and cached successfully!
   Model type : <class 'deepface.models.facial_recognition.Facenet.FaceNet128dClient'>

🧪 Running a quick sanity test with sample images...
✅ Sample images created in 'sample_images/' folder.

🎉 Training/setup complete! Model is ready.
   You can now run predict.py or main.py (FastAPI).

📁 Model weights are cached at: ~/.deepface/weights/


In [13]:
%run task2_predict.py

  Face Authentication - Predict Script

📁 Upload Image 1 (first face):


Saving happy-cheerful-african-millennial-man-looking-camera-home-american-face-head-shot-smiling-young-black-single-guy-136500293.webp to happy-cheerful-african-millennial-man-looking-camera-home-american-face-head-shot-smiling-young-black-single-guy-136500293.webp

📁 Upload Image 2 (second face):


Saving images.jpeg to images.jpeg

🔍 Comparing:
  Image 1: happy-cheerful-african-millennial-man-looking-camera-home-american-face-head-shot-smiling-young-black-single-guy-136500293.webp
  Image 2: images.jpeg

✅ Result          : DIFFERENT PERSON
   Verified        : False
   Similarity Score: 1.1965
   Threshold Used  : 0.4
   Face 1 BBox     : [0, 0, 799, 533]
   Face 2 BBox     : [137, 29, 56, 56]


In [14]:
!pip install pyngrok

In [17]:
from pyngrok import ngrok

# Paste your token from the dashboard (the one visible on your screen)
ngrok.set_auth_token("3EfFS1fynPvd04sq0sG9ZrBwUce_ykDKUhaFcp1zH4zoKXZd")

print("✅ ngrok token set successfully!")

✅ ngrok token set successfully!


In [18]:
import subprocess, threading, time

def run_server():
    subprocess.run(["uvicorn", "task2_main:app", "--host", "0.0.0.0", "--port", "8000"])

threading.Thread(target=run_server, daemon=True).start()
time.sleep(4)

url = ngrok.connect(8000)
print("="*50)
print(f"✅ API is LIVE!")
print(f"📖 Swagger UI  : {url}/docs")
print(f"🔗 API Base URL: {url}")
print("="*50)

✅ API is LIVE!
📖 Swagger UI  : NgrokTunnel: "https://account-untrained-worried.ngrok-free.dev" -> "http://localhost:8000"/docs
🔗 API Base URL: NgrokTunnel: "https://account-untrained-worried.ngrok-free.dev" -> "http://localhost:8000"


In [19]:
from google.colab import files
import requests as req

print("📁 Upload face image 1:")
u1 = files.upload()
img1 = list(u1.keys())[0]

print("📁 Upload face image 2:")
u2 = files.upload()
img2 = list(u2.keys())[0]

with open(img1, "rb") as a, open(img2, "rb") as b:
    response = req.post(
        "http://localhost:8000/verify",
        files={
            "image1": (img1, a, "image/jpeg"),
            "image2": (img2, b, "image/jpeg"),
        }
    )

result = response.json()
print("\n" + "="*45)
print(f"🎯 RESULT          : {result['result'].upper()}")
print(f"   Verified        : {result['verified']}")
print(f"   Similarity Score: {result['similarity_score']}")
print(f"   Threshold Used  : {result['threshold_used']}")
print(f"   Face 1 BBox     : {result['face1_bbox']}")
print(f"   Face 2 BBox     : {result['face2_bbox']}")
print("="*45)

📁 Upload face image 1:


Saving happy-cheerful-african-millennial-man-looking-camera-home-american-face-head-shot-smiling-young-black-single-guy-136500293.webp to happy-cheerful-african-millennial-man-looking-camera-home-american-face-head-shot-smiling-young-black-single-guy-136500293 (1).webp
📁 Upload face image 2:


Saving images.jpeg to images (1).jpeg

🎯 RESULT          : DIFFERENT PERSON
   Verified        : False
   Similarity Score: 1.1965
   Threshold Used  : 0.4
   Face 1 BBox     : [0, 0, 799, 533]
   Face 2 BBox     : [137, 29, 56, 56]


In [21]:
from google.colab import files
import requests as req

# Upload any 2 face photos (same person or different)
print("📁 Upload face image 1:")
u1 = files.upload()
img1 = list(u1.keys())[0]

print("📁 Upload face image 2:")
u2 = files.upload()
img2 = list(u2.keys())[0]

# Call the API
with open(img1, "rb") as a, open(img2, "rb") as b:
    response = req.post(
        "http://localhost:8000/verify",
        files={
            "image1": (img1, a, "image/jpeg"),
            "image2": (img2, b, "image/jpeg"),
        }
    )

result = response.json()
print("\n" + "="*45)
print(f"🎯 RESULT          : {result['result'].upper()}")
print(f"   Verified        : {result['verified']}")
print(f"   Similarity Score: {result['similarity_score']}")
print(f"   Threshold Used  : {result['threshold_used']}")
print(f"   Face 1 BBox     : {result['face1_bbox']}")
print(f"   Face 2 BBox     : {result['face2_bbox']}")
print("="*45)
print("\n✅ FastAPI is working perfectly!")


📁 Upload face image 1:


Saving WhatsApp Image 2026-06-04 at 14.06.13.jpeg to WhatsApp Image 2026-06-04 at 14.06.13.jpeg
📁 Upload face image 2:


Saving WhatsApp Image 2026-06-04 at 14.06.05.jpeg to WhatsApp Image 2026-06-04 at 14.06.05 (2).jpeg

🎯 RESULT          : DIFFERENT PERSON
   Verified        : False
   Similarity Score: 0.8531
   Threshold Used  : 0.4
   Face 1 BBox     : [40, 703, 1020, 1020]
   Face 2 BBox     : [194, 600, 957, 969]

✅ FastAPI is working perfectly!
